# Tích hợp dữ liệu phục vụ dự báo CPI giao thông

Notebook này thực hiện tích hợp các bộ dữ liệu đã được tiền xử lý về cùng tần suất tháng, bao gồm CPI nhóm Giao thông, giá xăng RON95, giá dầu Diesel, giá dầu thô Brent, WTI và tỷ giá USD/VND.

In [49]:
import pandas as pd

In [50]:
cpi = pd.read_csv("d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/interim/cpi_transport_monthly.csv")

fuel = pd.read_csv("d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/interim/fuel_prices_monthly.csv")

brent = pd.read_csv("d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/interim/brent_monthly.csv")

wti = pd.read_csv("d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/interim/wti_monthly.csv")

usd_vnd = pd.read_csv("d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/interim/usd_vnd_monthly.csv")

print("CPI:", cpi.columns.tolist())
print("Fuel:", fuel.columns.tolist())
print("Brent:", brent.columns.tolist())
print("WTI:", wti.columns.tolist())
print("USD/VND:", usd_vnd.columns.tolist())

CPI: ['MonthYear', 'CPI']
Fuel: ['MonthYear', 'Diesel', 'RON95']
Brent: ['MonthYear', 'Brent']
WTI: ['MonthYear', 'WTI']
USD/VND: ['MonthYear', 'USD_VND']


In [51]:
cpi["MonthYear"] = pd.to_datetime(cpi["MonthYear"]).dt.to_period("M")
fuel["MonthYear"] = pd.to_datetime(fuel["MonthYear"]).dt.to_period("M")
brent["MonthYear"] = pd.to_datetime(brent["MonthYear"]).dt.to_period("M")
wti["MonthYear"] = pd.to_datetime(wti["MonthYear"]).dt.to_period("M")
usd_vnd["MonthYear"] = pd.to_datetime(usd_vnd["MonthYear"]).dt.to_period("M")

print(cpi["MonthYear"].dtype)
print(fuel["MonthYear"].dtype)
print(brent["MonthYear"].dtype)
print(wti["MonthYear"].dtype)
print(usd_vnd["MonthYear"].dtype)

period[M]
period[M]
period[M]
period[M]
period[M]


In [52]:
dataset = (
    cpi
    .merge(fuel, on="MonthYear", how="left")
    .merge(brent, on="MonthYear", how="left")
    .merge(wti, on="MonthYear", how="left")
    .merge(usd_vnd, on="MonthYear", how="left")
)

dataset.head()

,MonthYear,CPI,Diesel,RON95,Brent,WTI,USD_VND
0,2011-01,NaN,NaN,NaN,NaN,NaN,NaN
1,2011-02,NaN,NaN,NaN,NaN,NaN,NaN
2,2011-03,NaN,NaN,NaN,NaN,NaN,NaN
3,2011-04,NaN,NaN,NaN,NaN,NaN,NaN
4,2011-05,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
dataset.shape

(168, 7)

In [54]:
dataset.isna().sum()

MonthYear     0
CPI          10
Diesel        8
RON95         8
Brent         8
WTI           8
USD_VND       8
dtype: int64

## Bổ sung biến giả Tết Nguyên đán

Biến `Dummy_Tet` được sử dụng để phản ánh ảnh hưởng của dịp Tết Nguyên đán đến nhu cầu đi lại và CPI nhóm Giao thông.

- `Dummy_Tet = 1`: tháng chịu tác động chính của Tết Nguyên đán.
- `Dummy_Tet = 0`: các tháng còn lại.

In [55]:
tet_months = {
    2012: 1,
    2013: 2,
    2014: 1,
    2015: 2,
    2016: 2,
    2017: 1,
    2018: 2,
    2019: 2,
    2020: 1,
    2021: 2,
    2022: 2,
    2023: 1,
    2024: 2
}

dataset["Dummy_Tet"] = dataset["MonthYear"].apply(
    lambda x: 1 if tet_months.get(x.year) == x.month else 0
)

dataset.loc[
    dataset["Dummy_Tet"] == 1,
    ["MonthYear", "Dummy_Tet"]
]

,MonthYear,Dummy_Tet
12,2012-01,1
25,2013-02,1
36,2014-01,1
49,2015-02,1
61,2016-02,1
72,2017-01,1
85,2018-02,1
97,2019-02,1
108,2020-01,1
121,2021-02,1


In [56]:
dataset["MonthYear"].duplicated().sum()

np.int64(0)

## Bổ sung biến giả Covid-19

Biến `Dummy_Covid` được sử dụng để phản ánh giai đoạn dịch Covid-19 ảnh hưởng đến hoạt động đi lại và CPI nhóm Giao thông.

- `Dummy_Covid = 1`: từ tháng 01/2020 đến tháng 10/2021.
- `Dummy_Covid = 0`: các tháng còn lại.

In [57]:
dataset["Dummy_Covid"] = (
    (dataset["MonthYear"] >= pd.Period("2020-01", freq="M")) &
    (dataset["MonthYear"] <= pd.Period("2021-10", freq="M"))
).astype(int)

dataset.loc[
    dataset["Dummy_Covid"] == 1,
    ["MonthYear", "Dummy_Covid"]
]

,MonthYear,Dummy_Covid
108,2020-01,1
109,2020-02,1
110,2020-03,1
111,2020-04,1
112,2020-05,1
113,2020-06,1
114,2020-07,1
115,2020-08,1
116,2020-09,1
117,2020-10,1


In [58]:
dataset = dataset[
    [
        "MonthYear",
        "CPI",
        "RON95",
        "Diesel",
        "Brent",
        "WTI",
        "USD_VND",
        "Dummy_Tet",
        "Dummy_Covid"
    ]
]

dataset.head()

,MonthYear,CPI,RON95,Diesel,Brent,WTI,USD_VND,Dummy_Tet,Dummy_Covid
0,2011-01,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,2011-02,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,2011-03,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,2011-04,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,2011-05,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [62]:
dataset = dataset[
    dataset["MonthYear"] >= pd.Period("2011-09", freq="M")
].copy()
dataset.head(10)

,MonthYear,CPI,RON95,Diesel,Brent,WTI,USD_VND,Dummy_Tet,Dummy_Covid
8,2011-09,NaN,21800.00,21100.00,112.83,85.52,20628.0,0,0
9,2011-10,NaN,21800.00,20400.00,109.55,86.32,20713.0,0,0
10,2011-11,-0.01,21800.00,20400.00,110.77,97.16,20803.0,0,0
11,2011-12,0.16,21800.00,20400.00,107.87,98.56,20813.6,0,0
12,2012-01,0.66,21800.00,20400.00,110.69,100.27,20828.0,1,0
13,2012-02,0.23,21800.00,20400.00,119.33,102.20,20828.0,0,0
14,2012-03,1.08,23090.32,21206.45,125.45,106.16,20828.0,0,0
15,2012-04,2.67,23400.00,21400.00,119.75,103.32,20828.0,0,0
16,2012-05,1.32,23522.58,21432.26,110.34,94.66,20828.0,0,0
17,2012-06,-1.64,22326.67,20506.67,95.16,82.30,20828.0,0,0


In [63]:
dataset.to_csv(
    "d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/processed/model_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)